# Resonant cyclothron scattering

We study the resonant cyclothron scattering and how it modifies a black-body spectrum at source. We assume the 1D model from [Lyutikov and Gavrill 2006](https://ui.adsabs.harvard.edu/abs/2006MNRAS.368..690L/abstract) (See also [Rea et al. 2008](https://ui.adsabs.harvard.edu/abs/2008ApJ...686.1245R/abstract)).

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import scipy.integrate as integrate
import scipy.special as scsp

from scipy.integrate import trapz, quad
import pypopsyn.simulator.basics.constants as const
import pypopsyn.simulator.multiband_emission.emission_xray as xem
import pypopsyn.simulator.interstellar_medium.nh_model as nhm
import pypopsyn.simulator.interstellar_medium.xray_abs_cross_section as xabs
import utilities.plot_settings

In [ ]:
def resonant_cyclothron_scat_spectrum_detail(
    E: np.ndarray,
    E_0: np.ndarray,
    tau_0: np.ndarray,
    beta_T: np.ndarray,
    I_ph_source: np.ndarray,
    n_reflections: int,
) -> np.ndarray:
    """
    Compute the spectrum resulting from different reflections and transmissions due to resonant cyclothron scattering (RCS) given a source intensity spectrum
    (see Lyutikov and Gavrill 2006).

    Args:
        E (np.ndarray): array of energies in [eV] of the transmitted intensity.
        E_0 (np.ndarray): array of energies in [eV] of the source intensity.
        tau_0 (np.ndarray): array of optical depths tau_0 (see eq. 2 in Lyutikov and Gavrill 2006).
        beta_T (np.ndarray): array of thermal velocities for the electrons/positrons in units of the speed of light.
        I_ph_source (np.ndarray): intensity of the source in [ph cm^-2 s^-1 eV^-1 sterad^-1].
        n_reflections (int): number of reflections (6 reflections guarantees convergence of the final spectrum,
                             see Lyutikov and Gavrill 2006).

    Returns:
        (np.ndarray): resonant cyclothron scattering spectrum intensity in [ph cm^-2 s^-1 eV^-1 sterad^-1].
    """
    rcs_spectrum = np.zeros((len(tau_0), n_reflections + 1, len(E)))

    # Convert photon energy in eV into omega frequencies in Hz.
    freq_0 = E_0 * const.EV_TO_ERG / const.H
    omega_0 = 2.0 * np.pi * freq_0

    freq = E * const.EV_TO_ERG / const.H
    omega = 2.0 * np.pi * freq

    # Compute the transmission and reflection probabilities.
    p_trans, p_refl = xem.trans_reflect_prob(omega, omega_0, tau_0, beta_T)

    # Compute the RCS spectrum by considering multiple reflections and transmissions.
    # (see eq. 42 in Lyutikov and Gavrill 2006).
    I_ph = I_ph_source[:, np.newaxis, :]
    I_ph_trans = trapz((I_ph * p_trans), omega_0, axis=2)
    rcs_spectrum[:, 0, :] = rcs_spectrum[:, 0, :] + I_ph_trans

    for i in range(1, n_reflections + 1):
        I_ph_reflect = trapz((I_ph * p_refl), omega_0, axis=2)
        I_ph_reflect = I_ph_reflect[:, np.newaxis, :]
        I_ph_trans_refl = trapz((I_ph_reflect * p_trans), omega_0, axis=2)
        rcs_spectrum[:, i, :] = rcs_spectrum[:, i-1, :] + I_ph_trans_refl

        I_ph = I_ph_trans_refl[:, np.newaxis, :]

    return rcs_spectrum

For this example we consider a black-body spectrum at source with $k_{\rm B} T = 1 \, keV$, an average plasma thermal velocity $\beta_{\rm T} = 0.3$ and resonant optical depth values of $\tau_{\rm res} = 0.2, 1, 2, 6, 10, 20$ to reproduce the first plot in Fig. 2 in [Gullon et al. 2015](https://ui.adsabs.harvard.edu/abs/2015MNRAS.454..615G/abstract).

In [ ]:
T = np.array([1000 * const.EV_TO_ERG / const.K_B])
tau_res = np.array([0.2, 1., 2., 6., 10., 20.])
beta_T = np.array([0.3, 0.3, 0.3, 0.3, 0.3, 0.3])
tau_0 = tau_res / 2.

res = 1000
E_0 = np.logspace(1., np.log10(20000), res)
freq_0 = E_0 * const.EV_TO_ERG / const.H
omega_0 = 2. * np.pi * freq_0

E = E_0
freq = E * const.EV_TO_ERG / const.H
omega = 2. * np.pi * freq

# Index to select a specific value of tau_res, choose between 0 and 5.
index = 4

First let's plot the transmission and reflection probabilities as a function of the frequency ratio $\omega / \omega_0$ to reproduce Fig. 2 in [Lyutikov and Gavrill 2006](https://ui.adsabs.harvard.edu/abs/2006MNRAS.368..690L/abstract).

In [ ]:
# Reshape omega to make it compatible for broadcasting.
omega_reshape = omega[:, np.newaxis]
omega_ratio = omega_reshape / omega_0

p_trans, p_refl = xem.trans_reflect_prob(omega, omega_0, tau_0, beta_T)

In [ ]:
p_tot_trans = trapz(p_trans, omega, axis=1)
p_tot_refl = trapz(p_refl, omega, axis=1)

# Total reflection probabilities.
print((1 - np.exp(-tau_0[index]))/2.)

print(p_tot_trans[index])
print(p_tot_refl[index])

# Verify that transmission and reflection probabilities sum up to 1.
print(p_tot_trans[index] + p_tot_refl[index])

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

#ax.set_xscale('log') 
#ax.set_yscale('log')
ax.set_xlim(0.2,2.2) 
#ax.set_ylim(0.,2.e-18) 
ax.set_xlabel(r'$\omega / \omega_0$')
ax.set_ylabel(r'$\mathcal{P}$')

ax.plot( 
    omega_ratio[:,500].flatten(),
    p_trans[index, :,500].flatten(),
    linestyle='-',
    linewidth=4,
    color="black",
    rasterized=True,
    label=r"Transmission probability",
)
ax.plot( 
    omega_ratio[:,500].flatten(),
    p_refl[index, :, 500].flatten(),
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
    label=r"Reflection probability",
)

plt.legend(frameon=False, loc=0)
plt.grid()

Compute the RCS spectrum.

In [ ]:
I_source = xem.blackbody_intensity_spectrum(E_0, T)
I_ph_source = I_source / (E_0 * const.EV_TO_ERG)
I_rcs = xem.resonant_cyclothron_scat_spectrum(
    E, 
    E_0, 
    tau_0, 
    beta_T, 
    I_ph_source, 
    n_reflections=6
) * (E_0 * const.EV_TO_ERG)

I_rcs_detail = resonant_cyclothron_scat_spectrum_detail(
    E, 
    E_0, 
    tau_0, 
    beta_T, 
    I_ph_source, 
    n_reflections=6
) * (E_0 * const.EV_TO_ERG)

This plot shows that after 6 reflection + transmission contributions the spectrum already converges quite well.

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

#ax.set_xscale('log') 
#ax.set_yscale('log')
ax.set_xlim(0.1,10.) 
#ax.set_ylim(1.e-1,1.e20) 
ax.set_xlabel(r'Energy [keV]')
ax.set_ylabel(r'$I(E)$ [erg cm$^{-2}$ s$^{-1}$ eV$^{-1}$ sterad$^{-1}$]')

ax.plot( 
    E*1.e-3,
    I_source.flatten(),
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
    label=r"BB intensity",
)
for i in range(I_rcs_detail.shape[1]):
    ax.plot( 
        E*1.e-3,
        I_rcs_detail[index, i, :],
        linestyle='-',
        linewidth=4,
        color="tab:red",
        alpha=0.3,
        rasterized=True,
    )
ax.plot( 
    E*1.e-3,
    I_rcs[index, :].flatten(),
    linestyle='-',
    linewidth=4,
    color="tab:red",
    rasterized=True,
    label=r"RCS spectrum",
)
plt.legend(frameon=False, loc=0)
plt.grid()

The following plot tries to reproduce the first plot in Fig. 2 in [Gullon et al. 2015](https://ui.adsabs.harvard.edu/abs/2015MNRAS.454..615G/abstract). The units are different since we are plotting the intensity at the source not the flux observed on Earth. Also notice that in their legend they are mistakenly reporting the values of $tau_0$ instead of $\tau_{\rm res}$.

In [ ]:
fig, ax = plt.subplots(figsize=(12,10))

colors = plt.get_cmap("viridis", len(tau_res))
norm = mpl.colors.Normalize(vmin=np.min(tau_res), vmax=np.max(tau_res)+1)
sm = plt.cm.ScalarMappable(norm=norm, cmap=colors)
sm.set_array([])

#ax.set_xscale('log') 
#ax.set_yscale('log')
ax.set_xlim(0.1,10.) 
#ax.set_ylim(1.e-1,1.e20) 
ax.set_xlabel(r'Energy [keV]')
ax.set_ylabel(r'$I(E)$ [erg cm$^{-2}$ s$^{-1}$ eV$^{-1}$ sterad$^{-1}$]')

ax.plot( 
    E*1.e-3,
    I_source.flatten(),
    linestyle='--',
    linewidth=4,
    color="black",
    rasterized=True,
    label=r"BB intensity",
)
ax.plot( 
    E*1.e-3,
    I_rcs[0, :].flatten(),
    linestyle='-',
    linewidth=4,
    color=colors(0),
    rasterized=True,
    label=r"$\tau_{\rm res} = 0.2$",
)
ax.plot( 
    E*1.e-3,
    I_rcs[1, :].flatten(),
    linestyle='-',
    linewidth=4,
    color=colors(1),
    rasterized=True,
    label=r"$\tau_{\rm res} = 1$",
)
ax.plot( 
    E*1.e-3,
    I_rcs[2, :].flatten(),
    linestyle='-',
    linewidth=4,
    color=colors(2),
    rasterized=True,
    label=r"$\tau_{\rm res} = 2$",
)
ax.plot( 
    E*1.e-3,
    I_rcs[3, :].flatten(),
    linestyle='-',
    linewidth=4,
    color=colors(3),
    rasterized=True,
    label=r"$\tau_{\rm res} = 6$",
)
ax.plot( 
    E*1.e-3,
    I_rcs[4, :].flatten(),
    linestyle='-',
    linewidth=4,
    color=colors(4),
    rasterized=True,
    label=r"$\tau_{\rm res} = 10$",
)
ax.plot( 
    E*1.e-3,
    I_rcs[5, :].flatten(),
    linestyle='-',
    linewidth=4,
    color=colors(5),
    rasterized=True,
    label=r"$\tau_{\rm res} = 20$",
)
plt.legend(frameon=False, loc=0)
plt.grid()